In [ ]:

# %% [markdown]
# # 02 — ExponentiatedGradient with FPR parity (Zien Xu, W2)
# Output: preds/preds_expgrad.parquet, models/expgrad_components/ , models/expgrad_weights.csv
#
# WHY THE WRAPPER BELOW EXISTS:
# fairlearn's reduction enforces the constraint on each base estimator's predict() (0/1).
# XGBClassifier.predict() thresholds at 0.5. With 1.1% fraud, almost nothing crosses 0.5,
# so every component predicts "not fraud" and the FPR-parity constraint is satisfied trivially
# (0 == 0) while the model does nothing. We therefore make predict() threshold at the
# operating point we actually deploy: 5% FPR.

In [ ]:
# %% 1. Setup
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append('/content/drive/MyDrive/CS3268_Project/src')
import time, numpy as np, pandas as pd
!pip install fairlearn
from sklearn.base import BaseEstimator, ClassifierMixin
from fairlearn.reductions import ExponentiatedGradient, FalsePositiveRateParity
from common import (load_split, features, threshold_at_fpr, save_preds,
                    train_model, BEST_PARAMS, SEED, LABEL, ROOT)

tr, va, te = load_split("train"), load_split("val"), load_split("test")
Xtr, ytr, gtr = features(tr), tr[LABEL], tr["age_group"]
Xva, yva = features(va), va[LABEL]
Xte, yte = features(te), te[LABEL]

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 5.5 MB/s eta 0:00:00


In [ ]:
# %% 2. Base estimator: XGBoost whose predict() fires at 5% FPR, not 0.5
class FixedFPRXGB(BaseEstimator, ClassifierMixin):
    def __init__(self, fpr=0.05, params=None):
        self.fpr, self.params = fpr, params

    def fit(self, X, y, sample_weight=None):
        self.classes_ = np.array([0, 1])
        self.model_ = train_model(X, y, sample_weight=sample_weight,
                                  params=self.params or BEST_PARAMS)
        s = self.model_.predict_proba(X)[:, 1]
        self.cut_ = threshold_at_fpr(s, y, self.fpr)   # in-sample cutoff (caveat: optimistic)
        return self

    def score_(self, X):
        return self.model_.predict_proba(X)[:, 1]

    def predict(self, X):
        return (self.score_(X) >= self.cut_).astype(int)

In [ ]:
# %% 3. Timing probe — run ONE fit before committing to the full reduction
t0 = time.time()
FixedFPRXGB().fit(Xtr, ytr)
one_fit = time.time() - t0
print(f"one fit: {one_fit:.1f}s  -> ~{one_fit*40/60:.0f} min for 40 iterations")
# If that projection is over ~45 min, subsample honest rows (cell 3b) and rerun this probe.

# %% 3b. OPTIONAL fallback: keep all fraud, subsample honest rows, reweight back
SUBSAMPLE = False
if SUBSAMPLE:
    rng = np.random.default_rng(SEED)
    fraud_idx = np.flatnonzero(ytr.values == 1)
    honest_idx = np.flatnonzero(ytr.values == 0)
    keep_honest = rng.choice(honest_idx, size=150_000, replace=False)
    idx = np.sort(np.concatenate([fraud_idx, keep_honest]))
    w_honest = len(honest_idx) / len(keep_honest)
    sw = np.where(ytr.values[idx] == 1, 1.0, w_honest)
    Xfit, yfit, gfit = Xtr.iloc[idx], ytr.iloc[idx], gtr.iloc[idx]
else:
    Xfit, yfit, gfit, sw = Xtr, ytr, gtr, None

one fit: 26.2s  -> ~17 min for 40 iterations


In [ ]:
# %% 4. Run the reduction
eg = ExponentiatedGradient(
    estimator=FixedFPRXGB(fpr=0.05),
    constraints=FalsePositiveRateParity(difference_bound=0.01),
    max_iter=40,
)
t0 = time.time()
eg.fit(Xfit, yfit, sensitive_features=gfit)
print(f"fitted in {(time.time()-t0)/60:.1f} min")

w = np.asarray(eg.weights_, dtype=float)
print("components:", len(eg.predictors_), " non-zero weights:", (w > 1e-8).sum())


fitted in 9.3 min
components: 5  non-zero weights: 2


In [ ]:
# %% 5. Continuous mixture score = weighted average of component PROBABILITIES
# (not eg.predict(), which samples one component at random, and not _pmf_predict(),
#  which averages 0/1 votes and gives a coarse score unusable at a 5% cutoff)
def mixture_score(X):
    s = np.zeros(len(X))
    for wi, p in zip(w, eg.predictors_):
        if wi > 1e-8:
            s += wi * p.score_(X)
    return s / w[w > 1e-8].sum()

s_va, s_te = mixture_score(Xva), mixture_score(Xte)
print("val score spread:", np.round(np.quantile(s_va, [0, .5, .95, .99, 1]), 4))
# Sanity: this must look continuous. If it takes only a handful of distinct values, stop and debug.

val score spread: [0.     0.0077 0.1204 0.2822 0.9122]


In [ ]:
# %% 6. Cutoff on val, write the predictions file
cut = threshold_at_fpr(s_va, yva, 0.05)
vt = pd.concat([va, te], ignore_index=True)
path = save_preds("expgrad", vt, np.concatenate([s_va, s_te]), cut)
print("saved", path, "cutoff =", round(cut, 5))

saved /content/drive/MyDrive/CS3268/preds/preds_expgrad.parquet cutoff = 0.11201


In [ ]:
# %% 7. Save components for Jianrong's SHAP in W4
import os, json
os.makedirs(f"{ROOT}/models/expgrad_components", exist_ok=True)
kept = [(i, wi) for i, wi in enumerate(w) if wi > 1e-8]
for i, _ in kept:
    eg.predictors_[i].model_.save_model(f"{ROOT}/models/expgrad_components/comp_{i}.json")
pd.DataFrame(kept, columns=["component", "weight"]).to_csv(
    f"{ROOT}/models/expgrad_weights.csv", index=False)
print("saved", len(kept), "components")

saved 2 components


In [ ]:
# %% 8. Quick read before Chloe's official numbers
p = pd.read_parquet(path)
p["split"] = np.where(p.month == 5, "val", "test")
def summary(d):
    return pd.Series({"FPR": d[d.y_true == 0].decision.mean(),
                      "recall": d[d.y_true == 1].decision.mean()})
print(p.groupby(["split", "age_group"]).apply(summary, include_groups=False).round(4))

                    FPR  recall
split age_group                
test  0          0.0708  0.6003
      1          0.0132  0.2649
val   0          0.0586  0.5494
      1          0.0083  0.2085
